In [ ]:
import os, math, copy, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# Hard constraints
MAX_STEPS = 50_000
BATCH_SIZE = 128
LR = 2e-4
Z_DIM = 128
NUM_CLASSES = 100
FID_SAMPLES = 10_000


In [ ]:
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,)*3, (0.5,)*3),
])

train_ds = torchvision.datasets.CIFAR100(
    root="data", train=True, download=True, transform=transform
)
test_ds = torchvision.datasets.CIFAR100(
    root="data", train=False, download=True, transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=2
)
test_loader = torch.utils.data.DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

def cycle(loader):
    while True:
        for x in loader:
            yield x

train_iter = iter(cycle(train_loader))

print("Train images:", len(train_ds))
print("Test images:", len(test_ds))


In [ ]:
from torch.nn.utils import spectral_norm

class ConditionalBatchNorm2d(nn.Module):
    def __init__(self, c, ncls):
        super().__init__()
        self.bn = nn.BatchNorm2d(c, affine=False)
        self.emb = nn.Embedding(ncls, c * 2)
        self.emb.weight.data[:, :c].normal_(1, 0.02)
        self.emb.weight.data[:, c:].zero_()

    def forward(self, x, y):
        g, b = self.emb(y).chunk(2, 1)
        return g[:, :, None, None] * self.bn(x) + b[:, :, None, None]

class SelfAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.q = nn.Conv2d(c, c//8, 1)
        self.k = nn.Conv2d(c, c//8, 1)
        self.v = nn.Conv2d(c, c, 1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B,C,H,W = x.shape
        q = self.q(x).view(B, -1, H*W).permute(0,2,1)
        k = self.k(x).view(B, -1, H*W)
        attn = torch.softmax(torch.bmm(q, k), dim=-1)
        v = self.v(x).view(B, C, H*W)
        out = torch.bmm(v, attn.permute(0,2,1)).view(B,C,H,W)
        return x + self.gamma * out

class GBlock(nn.Module):
    def __init__(self, ci, co, ncls):
        super().__init__()
        self.cbn1 = ConditionalBatchNorm2d(ci, ncls)
        self.cbn2 = ConditionalBatchNorm2d(co, ncls)
        self.c1 = nn.Conv2d(ci, co, 3, 1, 1)
        self.c2 = nn.Conv2d(co, co, 3, 1, 1)
        self.skip = nn.Conv2d(ci, co, 1)

    def forward(self, x, y):
        h = F.relu(self.cbn1(x, y))
        h = F.interpolate(h, scale_factor=2)
        h = self.c1(h)
        h = F.relu(self.cbn2(h, y))
        h = self.c2(h)
        s = self.skip(F.interpolate(x, scale_factor=2))
        return h + s

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(Z_DIM, 160 * 4 * 4)

        self.b1 = GBlock(160, 80, NUM_CLASSES)   # 4×4 → 8×8
        self.b2 = GBlock(80, 40, NUM_CLASSES)    # 8×8 → 16×16
        self.attn = SelfAttention(40)            # keep attention
        self.b3 = GBlock(40, 20, NUM_CLASSES)    # 16×16 → 32×32

        self.bn = nn.BatchNorm2d(20)
        self.out = nn.Conv2d(20, 3, 3, 1, 1)

    def forward(self, z, y):
        h = self.fc(z).view(-1, 160, 4, 4)
        h = self.b1(h, y)
        h = self.b2(h, y)
        h = self.attn(h)
        h = self.b3(h, y)
        h = F.relu(self.bn(h))
        return torch.tanh(self.out(h))

class DBlock(nn.Module):
    def __init__(self, ci, co, down=True):
        super().__init__()
        self.c1 = spectral_norm(nn.Conv2d(ci, co, 3, 1, 1))
        self.c2 = spectral_norm(nn.Conv2d(co, co, 3, 1, 1))
        self.skip = spectral_norm(nn.Conv2d(ci, co, 1))
        self.down = down

    def forward(self, x):
        h = F.relu(self.c1(x))
        h = self.c2(h)
        if self.down: h = F.avg_pool2d(h,2)
        s = self.skip(x)
        if self.down: s = F.avg_pool2d(s,2)
        return h + s

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.b1 = DBlock(3, 32)
        self.b2 = DBlock(32, 64)
        self.b3 = DBlock(64, 128)
        self.fc = spectral_norm(nn.Linear(128,1))
        self.emb = spectral_norm(nn.Embedding(NUM_CLASSES,128))

    def forward(self, x, y):
        h = self.b1(x)
        h = self.b2(h)
        h = self.b3(h)
        h = F.relu(h).sum([2,3])
        return self.fc(h) + (self.emb(y)*h).sum(1,keepdim=True)

G = Generator().to(DEVICE)
D = Discriminator().to(DEVICE)
ema_G = copy.deepcopy(G)

total_params = sum(p.numel() for p in G.parameters()) + sum(p.numel() for p in D.parameters())
print("Total parameters:", total_params)
assert total_params < 1_000_000


In [ ]:

from torchvision.utils import make_grid, save_image
import os

os.makedirs("outputs", exist_ok=True)
os.makedirs("real", exist_ok=True)
os.makedirs("gen", exist_ok=True)

def d_loss(r,f): 
    return (F.relu(1-r)+F.relu(1+f)).mean()

def g_loss(f): 
    return -f.mean()

g_opt = torch.optim.Adam(G.parameters(), lr=LR, betas=(0.0,0.9))
d_opt = torch.optim.Adam(D.parameters(), lr=LR, betas=(0.0,0.9))

preview_z = torch.randn(64, Z_DIM, device=DEVICE)
preview_y = torch.randint(0, NUM_CLASSES, (64,), device=DEVICE)

print("Starting training loop...")
steps = 0

while steps < MAX_STEPS:
    x, y = next(train_iter)
    x, y = x.to(DEVICE), y.to(DEVICE)
    z = torch.randn(x.size(0), Z_DIM, device=DEVICE)

    # --- Discriminator update ---
    fake = G(z, y).detach()
    loss_d = d_loss(D(x, y), D(fake, y))
    d_opt.zero_grad()
    loss_d.backward()
    d_opt.step()

    # --- Generator update ---
    fake = G(z, y)
    loss_g = g_loss(D(fake, y))
    g_opt.zero_grad()
    loss_g.backward()
    g_opt.step()

    # --- EMA update ---
    with torch.no_grad():
        for pe, p in zip(ema_G.parameters(), G.parameters()):
            pe.mul_(0.999).add_(p, alpha=0.001)

    steps += 1

    # --- Periodic logging ---
    if steps % 1000 == 0:
        print(f"[Step {steps}/{MAX_STEPS}] "
              f"D loss: {loss_d.item():.3f}, "
              f"G loss: {loss_g.item():.3f}")

    # --- Visualise + save the SAME 64 samples ---
    if steps % 5000 == 0:
        print(f"Generating and saving samples at step {steps}...")
        with torch.no_grad():
            samples_64 = ema_G(preview_z, preview_y).cpu()

        grid = make_grid(samples_64, nrow=8)
        plt.imshow(grid.permute(1,2,0)*0.5+0.5)
        plt.axis("off")
        plt.show()

        save_image(
            samples_64 * 0.5 + 0.5,
            "outputs/generated_samples_64.png",
            nrow=8
        )

print("Training complete.")
print("Total optimisation steps:", steps)


print("Generating latent-space interpolations...")

with torch.no_grad():
    z = torch.randn(16, Z_DIM, device=DEVICE)
    y = torch.randint(0, NUM_CLASSES, (16,), device=DEVICE)

    interp = []
    for t in torch.linspace(0, 1, 8):
        interp.append(
            ema_G((1-t)*z[:8] + t*z[8:], y[:8]).cpu()
        )

    interp = torch.cat(interp)

grid = make_grid(interp, nrow=8)
plt.imshow(grid.permute(1,2,0)*0.5+0.5)
plt.axis("off")
plt.show()

save_image(
    interp * 0.5 + 0.5,
    "outputs/latent_interpolations.png",
    nrow=8
)

print("Interpolations saved to outputs/latent_interpolations.png")


print("Preparing images for FID computation...")

from cleanfid import fid

@torch.no_grad()
def sample(n):
    z = torch.randn(n, Z_DIM, device=DEVICE)
    y = torch.randint(0, NUM_CLASSES, (n,), device=DEVICE)
    return ema_G(z, y)

count = 0
while count < FID_SAMPLES:
    imgs = sample(BATCH_SIZE).cpu()
    for img in imgs:
        if count >= FID_SAMPLES:
            break
        save_image(img*0.5+0.5, f"gen/{count}.png")
        count += 1

print("Generated images for FID.")

count = 0
for x, _ in test_loader:
    for img in x:
        if count >= FID_SAMPLES:
            break
        save_image(img*0.5+0.5, f"real/{count}.png")
        count += 1
    if count >= FID_SAMPLES:
        break

print("Saved real images for FID.")
print("Computing FID...")

print("FID:", fid.compute_fid("real", "gen", mode="clean"))
